In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_annotated/checkpoints/pre_cell_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 0 ###

### cell 0 – optimized for cudf
from pathlib import Path
import os, re

base_input = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"
mobile_list = []
broadband_list = []

# precompile regex to extract year and quarter
pattern = re.compile(r"year_(\d+)_quarter_(\d+)\.csv")
for filepath in base_input.rglob("*.csv"):
    # read and convert to nullable dtypes on GPU
    df = pd.read_csv(filepath, thousands=',').convert_dtypes()
    # rename columns if needed
    df.rename(columns={'Number of Record': 'Number of Records'}, inplace=True)
    # extract year and quarter
    yr, qt = map(int, pattern.search(filepath.name).groups())
    df['Year'] = np.int64(yr)
    df['Quarter'] = np.int64(qt)
    # collect per category
    if 'mobile' in filepath.name:
        mobile_list.append(df)
    else:
        broadband_list.append(df)

# single concat per category on GPU
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

# ensure Year/Quarter are int64 and sort
Mobile_df = (Mobile_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)
Broadband_df = (Broadband_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)

# expand datasets
factor = 10
Mobile_df = pd.concat([Mobile_df] * (factor * 2), ignore_index=True)
Broadband_df = pd.concat([Broadband_df] * factor, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df.info()
Broadband_df.info()